# Example Brain MRI Preprocessing Pipeline

This notebook implements preprocessing steps for brain MRI using `ANTsPy` (Advanced Normalization Tools for Python).

## Setup

In [ ]:
!pip install antspyx nilearn

In [ ]:
import ants
import numpy as np
import matplotlib.pyplot as plt
from nilearn import datasets, plotting

In [ ]:
def plot_comparison(
    img_before: ants.ANTsImage,
    img_after: ants.ANTsImage,
    title_before: str,
    title_after: str,
    cut_coords: tuple[int, int, int],
    ):

    ants.image_write(img_before, '/tmp/before.nii.gz')
    ants.image_write(img_after, '/tmp/after.nii.gz')

    fig, axes = plt.subplots(2, 1, figsize=(10, 8))


    plotting.plot_anat(
        '/tmp/before.nii.gz', title=title_before, axes=axes[0], cut_coords=cut_coords
        )
    plotting.plot_anat(
        '/tmp/after.nii.gz', title=title_after, axes=axes[1], cut_coords=cut_coords
        )

    plt.show()

## Data Loading

Use a sample T1 image from the `haxby` dataset.

In [ ]:
dataset = datasets.fetch_haxby()
raw_path = dataset.anat[0]
raw_img = ants.image_read(raw_path)

In [ ]:
raw_img

In [ ]:
plotting.plot_anat(raw_path, title="Raw Input Image", cut_coords=(0,0,0))

Download the MNI152 template as a reference for registration.

In [ ]:
icbm_template_data = datasets.fetch_icbm152_2009()
mni_path = icbm_template_data['t1']
mni_img = ants.image_read(mni_path)

## Reorientation

Use `ants.reorient_image2` to align the image to the standard RAS (Right-Anterior-Superior) coordinate system used by MNI.

In [ ]:
reoriented_img = ants.reorient_image2(raw_img, orientation='RAS')

In [ ]:
reoriented_img

In [ ]:
cut_coords = reoriented_img.origin - 0.5 * (np.array(reoriented_img.shape) * np.array(reoriented_img.spacing))

## N4 Bias Field Correction

Fix errors from magnetic field distortions using N4 bias field correction.

In [ ]:
n4_img = ants.n4_bias_field_correction(reoriented_img)

In [ ]:
plot_comparison(reoriented_img, n4_img, "Reoriented", "N4 Corrected", cut_coords=cut_coords)

## Brain Extraction

Strip the skull with probability mask generation.

In [ ]:
mask = ants.get_mask(n4_img)
brain_extracted = ants.mask_image(n4_img, mask)

In [ ]:
plot_comparison(n4_img, brain_extracted, "Whole Head (N4)", "Brain Extracted", cut_coords=cut_coords)

## Registration into MNI Space

Use `ants.registration` with the SyN (Symmetric Normalization) transform, which includes Rigid, Affine, and Deformable steps.

In [ ]:
tx = ants.registration(
    fixed=mni_img,
    moving=brain_extracted,
)
warped_img = tx['warpedmovout']

In [ ]:
ants.image_write(warped_img, '/tmp/warped_subject.nii.gz')
ants.image_write(mni_img, '/tmp/mni_template.nii.gz')

In [ ]:
display = plotting.plot_anat('/tmp/warped_subject.nii.gz', title="Normalized to MNI Space")
display.add_edges('/tmp/mni_template.nii.gz')